# LangGraph & Open-Source Model Integration

This notebook demonstrates how to create a simple stateful agent workflow using **LangGraph** and a free **Open-Source GPT-OSS 20B Reasoning LLM** hosted on Pollinations.ai (requiring no API keys or setup).

We'll build a simple 2-node workflow:
1. **Joke Generator**: Generates a joke on a given topic.
2. **Joke Reviewer**: Evaluates the joke and refines/improves it.

In [ ]:
# Install the necessary libraries
%pip install langgraph requests python-dotenv -q --no-warn-script-location

In [ ]:
import os
import time
import requests
from dotenv import load_dotenv

# Load environment variables from the .env file (if any)
load_dotenv()

# Helper function to call the free open-source GPT-OSS model via pollinations.ai (no API key needed)
def call_open_source_model(prompt: str, system_prompt: str = "You are a creative writer assistant."):
    url = "https://text.pollinations.ai"
    payload = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
        "model": "openai-fast"
    }
    
    for attempt in range(3):
        try:
            response = requests.post(url, json=payload, timeout=30)
            if response.status_code == 200:
                class ModelResponse:
                    def __init__(self, text):
                        self.text = text
                return ModelResponse(response.text)
            else:
                print(f"⚠️ API error: {response.status_code}. Retrying...")
                time.sleep(2)
        except Exception as e:
            print(f"⚠️ Connection error: {e}. Retrying...")
            time.sleep(2)
            
    raise Exception("Failed to get response from pollinations.ai endpoint after 3 attempts.")

print("✅ Open-Source model helper initialized successfully!")

## Define the Graph State

In LangGraph, all data is passed through a shared state. We define the structure of our state using a standard Python `TypedDict`.

In [ ]:
from typing import TypedDict

class GraphState(TypedDict):
    topic: str
    joke: str
    feedback: str
    improved_joke: str

## Define the Nodes (Functions)

Nodes are Python functions that receive the current state, perform some action, and return a dictionary containing the state fields they want to update.

In [ ]:
def generate_joke_node(state: GraphState):
    print(f"🤖 [Generator Node] Generating a joke about '{state['topic']}'...")
    
    prompt = f"Write a funny, clean one-liner joke about: {state['topic']}"
    
    response = call_open_source_model(prompt)
    joke = response.text.strip()
    print(f"Generated Joke: {joke}\n")
    return {"joke": joke}

def review_joke_node(state: GraphState):
    print("🧐 [Reviewer Node] Reviewing the joke and making it better...")
    
    prompt = (
        f"Review this joke: \"{state['joke']}\"\n\n"
        "Provide a brief critique/feedback on how to make it funnier or punchier, "
        "and then write an improved version of the joke. Use the following format exactly:\n"
        "Feedback: [Your feedback here]\n"
        "Improved Joke: [Your improved joke here]"
    )
    
    response = call_open_source_model(prompt)
    response_text = response.text.strip()
    print(f"Reviewer Response:\n{response_text}\n")
    
    # Parse the response text
    feedback = ""
    improved_joke = ""
    for line in response_text.split("\n"):
        if line.startswith("Feedback:"):
            feedback = line.replace("Feedback:", "").strip()
        elif line.startswith("Improved Joke:"):
            improved_joke = line.replace("Improved Joke:", "").strip()
            
    # Fallback if parsing failed
    if not feedback:
        feedback = response_text
    if not improved_joke:
        improved_joke = state['joke']
        
    return {"feedback": feedback, "improved_joke": improved_joke}

## Build and Compile the Graph

We define the graph structure by adding our nodes and specifying the control flow using edges.

In [ ]:
from langgraph.graph import StateGraph, START, END

# Create the graph builder
builder = StateGraph(GraphState)

# Add our nodes to the graph
builder.add_node("generator", generate_joke_node)
builder.add_node("reviewer", review_joke_node)

# Define the workflow structure
builder.add_edge(START, "generator")
builder.add_edge("generator", "reviewer")
builder.add_edge("reviewer", END)

# Compile the graph into an executable application
app = builder.compile()
print("🎉 Graph compiled successfully!")

## Run the LangGraph Workflow

Now, we run the graph by invoking it with our initial state.

In [ ]:
# Invoke the graph with a topic
initial_input = {"topic": "artificial intelligence"}
result = app.invoke(initial_input)

print("\n" + "="*50)
print("🎯 FINAL RESULTS:")
print(f"Topic: {result['topic']}")
print(f"Original Joke: {result['joke']}")
print(f"Feedback: {result['feedback']}")
print(f"Improved Joke: {result['improved_joke']}")
print("="*50)